# Notebook 03 — Preprocessing & Feature Engineering

**Project:** IT Helpdesk Ticket Classification & Auto-Routing System  
**Input:** `data/raw/tickets.csv`  
**Outputs:** Processed feature matrices in `data/processed/`, fitted artifacts in `models/`

---

### What this notebook does
1. Load raw data and apply the text-cleaning pipeline
2. Stratified 70/15/15 train/val/test split
3. Build **Feature Set A: TF-IDF** (unigrams + bigrams, 5 000 features)
4. Build **Feature Set B: Word2Vec** (100-dim averaged document vectors)
5. Encode categorical targets (category, priority)
6. Save all processed features and fitted artifacts

In [1]:
# ── setup ──────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import (
    clean_text,
    build_tfidf,
    build_w2v,
    fit_label_encoder,
    encode_labels,
    make_splits,
    save_artifact,
)

PROCESSED = PROJECT_ROOT / 'data' / 'processed'
MODELS    = PROJECT_ROOT / 'models'
PROCESSED.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

print('Setup complete. Project root:', PROJECT_ROOT)

Setup complete. Project root: C:\Users\Pyramid\Desktop\NewProjects\it-ticket-classifier


---
## Step 1 — Load Raw Data

In [2]:
df = pd.read_csv(
    PROJECT_ROOT / 'data' / 'raw' / 'tickets.csv',
    parse_dates=['created_at', 'resolved_at']
)
print(f'Loaded {len(df):,} rows')
print(df[['ticket_id','ticket_text','category','priority']].head(3))

Loaded 4,000 rows
  ticket_id                                        ticket_text      category  \
0  TKT-0001  otp for two-factor auth not arriving on my reg...  Access/Login   
1  TKT-0002  not urgent - cannot access coud portal from of...       Network   
2  TKT-0003  several users on my floor - machine boots into...      Software   

  priority  
0      Low  
1      Low  
2   Medium  


---
## Step 2 — Text Cleaning

In [3]:
# Apply the full NLP pipeline from src/preprocessing.py
# Pipeline: lowercase → remove punctuation → normalise numbers → tokenise → stopword removal → stemming

print('Cleaning ticket texts...')
df['cleaned_text'] = df['ticket_text'].apply(clean_text)
print('Done.')

# Show before / after for 5 samples
sample_idx = df.sample(5, random_state=42).index
for i in sample_idx:
    print(f'\n[{df.loc[i,"category"]} | {df.loc[i,"priority"]}]')
    print(f'  RAW    : {df.loc[i,"ticket_text"]}')
    print(f'  CLEANED: {df.loc[i,"cleaned_text"]}')

Cleaning ticket texts...


Done.

[Software | Medium]
  RAW    : excel file says corrupted and ont open, need data urgently, this is blocking my work
  CLEANED: excel file say corrupt ont open need data urgent block work

[Hardware | Low]
  RAW    : front usb ports on desktop not working at all, thansk in advance
  CLEANED: front usb port desktop work thansk advanc

[Network | High]
  RAW    : cannot work at all, i - wifi password was reset and now 100 users in wing ground floor cannot connect, this is bloccking my work
  CLEANED: cannot work wifi password reset num user wing ground floor cannot connect blocck work

[Network | Critical]
  RAW    : whole team blocked - all printers offline on floor 100 after network maintenance last night
  CLEANED: whole team block printer offlin floor num network mainten last night

[Access/Login | High]
  RAW    : need shared drive \\SRV-DB01\Projects access for HR project, this is blocking my work
  CLEANED: need share drive srv db01 project access hr project block work


In [4]:
# ── Vocabulary size before and after cleaning ──────────────────────────────────
raw_vocab  = set(' '.join(df['ticket_text']).lower().split())
clean_vocab = set(' '.join(df['cleaned_text']).split())
print(f'Raw vocabulary size    : {len(raw_vocab):,} unique tokens')
print(f'Cleaned vocabulary size: {len(clean_vocab):,} unique tokens')
print(f'Reduction              : {(1 - len(clean_vocab)/len(raw_vocab))*100:.1f}%')
print('\nCleaning reduces vocab size substantially (stopwords + stemming)')

Raw vocabulary size    : 2,494 unique tokens
Cleaned vocabulary size: 2,005 unique tokens
Reduction              : 19.6%

Cleaning reduces vocab size substantially (stopwords + stemming)


---
## Step 3 — Train / Val / Test Split (70 / 15 / 15, stratified by category)

In [5]:
# make_splits() stratifies by category to preserve class ratios across all splits.
# This is critical for the minority classes (Database, Critical).
train_df, val_df, test_df = make_splits(df, target_col='category', seed=42)

print(f'\nSplit sizes: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')
print(f'Total: {len(train_df)+len(val_df)+len(test_df)} (matches original {len(df)})')

# Verify stratification — proportions should be nearly identical across splits
for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'\n{split_name} category %:')
    print((split_df['category'].value_counts(normalize=True)*100).round(1))

Split sizes  —  train: 2799, val: 601, test: 600
Train category distribution:
category
Software        684
Network         572
Access/Login    524
Hardware        440
Email           347
Database        232
Name: count, dtype: int64

Split sizes: train=2799, val=601, test=600
Total: 4000 (matches original 4000)

train category %:
category
Software        24.4
Network         20.4
Access/Login    18.7
Hardware        15.7
Email           12.4
Database         8.3
Name: proportion, dtype: float64

val category %:
category
Software        24.5
Network         20.5
Access/Login    18.8
Hardware        15.6
Email           12.3
Database         8.3
Name: proportion, dtype: float64

test category %:
category
Software        24.5
Network         20.5
Access/Login    18.7
Hardware        15.7
Email           12.3
Database         8.3
Name: proportion, dtype: float64


In [6]:
# Save the raw splits — needed in regression notebook for timestamp features
train_df.to_csv(PROCESSED / 'train.csv', index=False)
val_df.to_csv(  PROCESSED / 'val.csv',   index=False)
test_df.to_csv( PROCESSED / 'test.csv',  index=False)
print('Saved: data/processed/train.csv, val.csv, test.csv')

# Extract text lists for feature building
train_texts = train_df['cleaned_text'].tolist()
val_texts   = val_df['cleaned_text'].tolist()
test_texts  = test_df['cleaned_text'].tolist()

Saved: data/processed/train.csv, val.csv, test.csv


---
## Step 4a — Feature Set A: TF-IDF

**Design choices:**
- `ngram_range=(1,2)` — unigrams capture single keywords; bigrams capture phrases like "vpn not" or "access denied"
- `max_features=5000` — caps vocabulary to prevent memory issues and reduces noise from rare terms
- `sublinear_tf=True` — uses log(1+tf) instead of raw tf, reducing the dominance of very frequent terms
- `min_df=2` — ignores tokens appearing in fewer than 2 documents (likely typos or single-use terms)
- **Fit only on train** — fitting on val/test would leak information

In [7]:
X_train_tfidf, X_val_tfidf, X_test_tfidf, tfidf_vec = build_tfidf(
    train_texts, val_texts, test_texts,
    max_features=5000,
    ngram_range=(1, 2)
)

print(f'\nTF-IDF matrix density (train): {X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]):.4f}')
print('(Very sparse — expected for bag-of-words on short texts)')

TF-IDF vocabulary size : 2,764
Feature matrix shapes  : train=(2799, 2764), val=(601, 2764), test=(600, 2764)

TF-IDF matrix density (train): 0.0058
(Very sparse — expected for bag-of-words on short texts)


In [8]:
# ── Top 20 TF-IDF features (by mean weight across train corpus) ────────────────
import numpy as np
mean_tfidf = np.asarray(X_train_tfidf.mean(axis=0)).flatten()
top_idx    = mean_tfidf.argsort()[::-1][:20]
feature_names = np.array(tfidf_vec.get_feature_names_out())
print('Top 20 TF-IDF features by mean weight:')
for i, idx in enumerate(top_idx):
    print(f'  {i+1:2d}. {feature_names[idx]:<25} {mean_tfidf[idx]:.5f}')

Top 20 TF-IDF features by mean weight:
   1. num                       0.03183
   2. need                      0.02346
   3. work                      0.02057
   4. pleas                     0.01994
   5. access                    0.01892
   6. laptop                    0.01740
   7. team                      0.01633
   8. cannot                    0.01560
   9. updat                     0.01492
  10. urgent                    0.01471
  11. machin                    0.01390
  12. user                      0.01389
  13. help                      0.01380
  14. email                     0.01363
  15. desktop                   0.01221
  16. new                       0.01160
  17. window                    0.01148
  18. block                     0.01121
  19. password                  0.01080
  20. error                     0.01006


In [9]:
# ── Save TF-IDF matrices and vectorizer ────────────────────────────────────────
sp.save_npz(str(PROCESSED / 'X_train_tfidf.npz'), X_train_tfidf)
sp.save_npz(str(PROCESSED / 'X_val_tfidf.npz'),   X_val_tfidf)
sp.save_npz(str(PROCESSED / 'X_test_tfidf.npz'),  X_test_tfidf)
save_artifact(tfidf_vec, str(MODELS / 'tfidf_vectorizer.joblib'))
print('TF-IDF artifacts saved.')

Saved  : C:\Users\Pyramid\Desktop\NewProjects\it-ticket-classifier\models\tfidf_vectorizer.joblib
TF-IDF artifacts saved.


---
## Step 4b — Feature Set B: Word2Vec

**Design choices:**
- `vector_size=100` — 100-dim embeddings match what we will use to optionally initialise the LSTM embedding layer in Notebook 04
- `window=5` — context window of 5 words (standard for skip-gram)
- `min_count=2` — ignore very rare words
- **Averaged pooling** — each document becomes the mean of its word vectors; OOV words are skipped
- Train **only on train split** to prevent leakage

In [10]:
X_train_w2v, X_val_w2v, X_test_w2v, w2v_model = build_w2v(
    train_texts, val_texts, test_texts,
    vector_size=100,
    window=5,
    min_count=2,
    seed=42
)

Word2Vec vocab size : 752


Vector size         : 100
W2V matrix shapes   : train=(2799, 100), val=(601, 100), test=(600, 100)


In [11]:
# ── Word2Vec sanity check: similar words for domain terms ─────────────────────
test_words = ['vpn', 'outlook', 'databas', 'password', 'server']
for word in test_words:
    if word in w2v_model.wv:
        similar = w2v_model.wv.most_similar(word, topn=3)
        print(f'  Words similar to "{word}": {similar}')
    else:
        print(f'  "{word}" not in W2V vocab (may have been stemmed)')

  Words similar to "vpn": [('tunnel', 0.9736775159835815), ('system', 0.9696061015129089), ('drop', 0.964362382888794)]
  Words similar to "outlook": [('web', 0.9616267681121826), ('attach', 0.961229145526886), ('wehn', 0.9549797773361206)]
  Words similar to "databas": [('spike', 0.9957091212272644), ('latenc', 0.9935562610626221), ('min', 0.9852989912033081)]
  Words similar to "password": [('servic', 0.920329213142395), ('reset', 0.9137627482414246), ('forgot', 0.9073882102966309)]
  Words similar to "server": [('confer', 0.979190468788147), ('upload', 0.9790409803390503), ('complet', 0.9769748449325562)]


In [12]:
# ── Save Word2Vec matrices and model ─────────────────────────────────────────
np.save(str(PROCESSED / 'X_train_w2v.npy'), X_train_w2v)
np.save(str(PROCESSED / 'X_val_w2v.npy'),   X_val_w2v)
np.save(str(PROCESSED / 'X_test_w2v.npy'),  X_test_w2v)
w2v_model.save(str(MODELS / 'w2v_model.model'))
print('Word2Vec artifacts saved.')

Word2Vec artifacts saved.


---
## Step 5 — Label Encoding

In [13]:
# ── Category label encoder ─────────────────────────────────────────────────────
le_cat = fit_label_encoder(train_df['category'])

y_train_cat = encode_labels(train_df['category'], le_cat)
y_val_cat   = encode_labels(val_df['category'],   le_cat)
y_test_cat  = encode_labels(test_df['category'],  le_cat)

print('Category encoding:', dict(zip(le_cat.classes_, le_cat.transform(le_cat.classes_))))

Label encoder classes: ['Access/Login', 'Database', 'Email', 'Hardware', 'Network', 'Software']
Category encoding: {'Access/Login': 0, 'Database': 1, 'Email': 2, 'Hardware': 3, 'Network': 4, 'Software': 5}


In [14]:
# ── Priority label encoder ─────────────────────────────────────────────────────
le_pri = fit_label_encoder(train_df['priority'])

y_train_pri = encode_labels(train_df['priority'], le_pri)
y_val_pri   = encode_labels(val_df['priority'],   le_pri)
y_test_pri  = encode_labels(test_df['priority'],  le_pri)

print('Priority encoding:', dict(zip(le_pri.classes_, le_pri.transform(le_pri.classes_))))

Label encoder classes: ['Critical', 'High', 'Low', 'Medium']
Priority encoding: {'Critical': 0, 'High': 1, 'Low': 2, 'Medium': 3}


In [15]:
# ── Save label arrays and encoders ────────────────────────────────────────────
np.save(str(PROCESSED / 'y_train_cat.npy'), y_train_cat)
np.save(str(PROCESSED / 'y_val_cat.npy'),   y_val_cat)
np.save(str(PROCESSED / 'y_test_cat.npy'),  y_test_cat)

np.save(str(PROCESSED / 'y_train_pri.npy'), y_train_pri)
np.save(str(PROCESSED / 'y_val_pri.npy'),   y_val_pri)
np.save(str(PROCESSED / 'y_test_pri.npy'),  y_test_pri)

# Regression target — resolution time in hours
np.save(str(PROCESSED / 'y_train_reg.npy'), train_df['resolution_time_hours'].values)
np.save(str(PROCESSED / 'y_val_reg.npy'),   val_df['resolution_time_hours'].values)
np.save(str(PROCESSED / 'y_test_reg.npy'),  test_df['resolution_time_hours'].values)

save_artifact(le_cat, str(MODELS / 'label_encoder_category.joblib'))
save_artifact(le_pri, str(MODELS / 'label_encoder_priority.joblib'))

print('All label arrays and encoders saved.')

Saved  : C:\Users\Pyramid\Desktop\NewProjects\it-ticket-classifier\models\label_encoder_category.joblib
Saved  : C:\Users\Pyramid\Desktop\NewProjects\it-ticket-classifier\models\label_encoder_priority.joblib
All label arrays and encoders saved.


---
## Step 6 — Feature Comparison Summary

In [16]:
print('='*55)
print('FEATURE SUMMARY')
print('='*55)
print(f'TF-IDF  — shape: {X_train_tfidf.shape}  | sparse matrix')
print(f'Word2Vec — shape: {X_train_w2v.shape}  | dense matrix')
print()
print('Both sets saved to data/processed/')
print('Fitted artifacts saved to models/')
print()
print('Files created:')
for f in sorted(PROCESSED.iterdir()):
    print(f'  data/processed/{f.name}  ({f.stat().st_size//1024} KB)')
for f in sorted(MODELS.iterdir()):
    print(f'  models/{f.name}  ({f.stat().st_size//1024} KB)')

FEATURE SUMMARY
TF-IDF  — shape: (2799, 2764)  | sparse matrix
Word2Vec — shape: (2799, 100)  | dense matrix

Both sets saved to data/processed/
Fitted artifacts saved to models/

Files created:
  data/processed/test.csv  (130 KB)
  data/processed/train.csv  (600 KB)
  data/processed/val.csv  (129 KB)
  data/processed/X_test_tfidf.npz  (70 KB)
  data/processed/X_test_w2v.npy  (234 KB)
  data/processed/X_train_tfidf.npz  (315 KB)
  data/processed/X_train_w2v.npy  (1093 KB)
  data/processed/X_val_tfidf.npz  (68 KB)
  data/processed/X_val_w2v.npy  (234 KB)
  data/processed/y_test_cat.npy  (2 KB)
  data/processed/y_test_pri.npy  (2 KB)
  data/processed/y_test_reg.npy  (4 KB)
  data/processed/y_train_cat.npy  (11 KB)
  data/processed/y_train_pri.npy  (11 KB)
  data/processed/y_train_reg.npy  (21 KB)
  data/processed/y_val_cat.npy  (2 KB)
  data/processed/y_val_pri.npy  (2 KB)
  data/processed/y_val_reg.npy  (4 KB)
  models/label_encoder_category.joblib  (0 KB)
  models/label_encoder_priorit

---
## Summary

| Artifact | Location | Used in |
|---|---|---|
| `tfidf_vectorizer.joblib` | `models/` | Notebooks 04, 05, 06, 07; routing engine |
| `w2v_model.model` | `models/` | Notebooks 04 (LSTM init), 07 |
| `X_*_tfidf.npz` | `data/processed/` | Notebooks 04, 05 |
| `X_*_w2v.npy` | `data/processed/` | Notebook 04 |
| `y_*_cat.npy` | `data/processed/` | Notebooks 04, 07 |
| `y_*_pri.npy` | `data/processed/` | Notebooks 05, 07 |
| `y_*_reg.npy` | `data/processed/` | Notebook 06 |
| `label_encoder_*.joblib` | `models/` | All classification notebooks + routing engine |

**Next:** Run `04_category_classification.ipynb`